# MLP From Scratch
## Multi-Class MIDI Note Classification on GuitarSet

**Course:** CMOR 438 — Machine Learning  
**Dataset:** GuitarSet (`audio_hex_cln`, all 6 strings, 360 tracks — full dataset)  
**Task:** Given 18 audio features extracted from a 46 ms frame, classify the MIDI note being played — or identify the frame as silent.

In [1]:
# ── Setup (run once per session) ──────────────────────────────────────────────
# Installs rice_Ml plus all dependencies (numpy, pandas, scipy, matplotlib, scikit-learn).
!pip install -q git+https://github.com/seyaul/cmor438-s2026-final-project.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 1. Introduction

The **Multi-Layer Perceptron (MLP)** extends the Perceptron by stacking nonlinear layers, enabling it to learn complex, non-linearly-separable decision boundaries. Where the Perceptron failed to converge on voiced/silent detection (the data is not linearly separable), the MLP can learn the curved boundaries needed to distinguish individual MIDI notes.

### 1.1 From Perceptron to MLP

A single Perceptron computes $\hat{y} = \text{sign}(\mathbf{w}^\top \mathbf{x} + b)$ — one linear boundary. An MLP composes $L$ such transformations with nonlinear activations between them. By the **Universal Approximation Theorem** (Cybenko, 1989), a single hidden layer with enough units can approximate any continuous function on a compact domain.

### 1.2 Why this task is harder

We now classify **which MIDI note** is being played (or silence) — 13 classes instead of 2. The spectral features that distinguish one guitar note from another are subtler than the energy difference between silence and any note. Adjacent semitones share most of their harmonic content; only fine MFCC differences separate them.

## 2. Algorithm

### 2.1 Forward Pass

For $L$ layers with weight matrices $W^{(l)}$, biases $b^{(l)}$, and activation $\sigma$:

$$z^{(l)} = W^{(l)} a^{(l-1)} + b^{(l)}, \qquad a^{(l)} = \sigma\!\left(z^{(l)}\right)$$

The output layer produces a score vector $\hat{\mathbf{y}} \in \mathbb{R}^K$ for $K$ classes.

### 2.2 Loss

We minimise **Mean Squared Error** against one-hot targets $\mathbf{y} \in \{0,1\}^K$:

$$\mathcal{L} = \frac{1}{2K}\sum_{k=1}^{K}(\hat{y}_k - y_k)^2$$

### 2.3 Backpropagation

The chain rule propagates the loss gradient from output to input:

$$\delta^{(L)} = \nabla_{\hat{y}}\mathcal{L} \odot \sigma'\!\left(z^{(L)}\right)$$
$$\delta^{(l)} = \left(W^{(l+1)\top} \delta^{(l+1)}\right) \odot \sigma'\!\left(z^{(l)}\right)$$

Weight gradients: $\nabla_{W^{(l)}} \mathcal{L} = \delta^{(l)} a^{(l-1)\top}$

### 2.4 Mini-Batch SGD

$$W^{(l)} \leftarrow W^{(l)} - \eta \cdot \frac{1}{|\mathcal{B}|} \sum_{i \in \mathcal{B}} \nabla_{W^{(l)}} \mathcal{L}_i$$

where $\mathcal{B}$ is a randomly sampled mini-batch of size $B$.

## 3. Imports

In [2]:
import sys
from pathlib import Path

def _find_repo_root(marker: str = "pyproject.toml") -> Path | None:
    for candidate in [Path().resolve(), *Path().resolve().parents]:
        if (candidate / marker).exists():
            return candidate
    return None

REPO_ROOT = _find_repo_root()
if REPO_ROOT is not None:
    sys.path.insert(0, str(REPO_ROOT / "src"))
    print(f"Repo root: {REPO_ROOT}")
else:
    try:
        import rice_Ml as _check
        REPO_ROOT = Path(_check.__file__).resolve().parent.parent.parent
        print(f"rice_Ml installed at: {Path(_check.__file__).resolve()}")
    except ImportError:
        raise ImportError("rice_Ml not found. Run the setup cell above first.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier as SklearnMLP
from sklearn.preprocessing import StandardScaler as SklearnScaler

from rice_Ml.datasets import load_guitarset
from rice_Ml.supervised_ml import MLP
from rice_Ml.activations import ReLU
from rice_Ml.loss import MeanSquaredError
from rice_Ml.optimizers import SGD
from rice_Ml.preprocessing.scale import StandardScaler
from rice_Ml.model_selection.split import train_test_split, KFold
from rice_Ml.metrics import accuracy, precision, recall, f1_score

SEED = 42
rng = np.random.default_rng(SEED)

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})
print("Imports OK")

rice_Ml installed at: /usr/local/lib/python3.12/dist-packages/rice_Ml/__init__.py
Imports OK


## 4. Load Data

We use the **full GuitarSet dataset** — all 360 tracks × 6 strings (~5.6 M frames). Downloaded from GitHub Releases on first use (~855 MB), then cached at `~/.cache/rice_ml/`.

**Label construction:** We keep the **silence class** (midi_label = 0) plus the **top 12 most frequent MIDI notes**, giving **13 classes** total. Labels are remapped to contiguous integers 0–12.

| Class | Meaning |
|-------|---------|
| 0 | Silent frame |
| 1–12 | Top-12 MIDI notes (remapped from raw MIDI values) |

**Class balancing:** Silence accounts for ~82% of raw frames — a classifier can achieve 82% accuracy by predicting silence for every input, without learning any notes. To give the model balanced gradient signal, we undersample silence to **2× the most frequent voiced class** using `rice_Ml.preprocessing.undersample_majority`. This reduces the dataset from ~4.9 M → ~850 K rows and silence from 82% → ~22%, while keeping all voiced frames intact.

In [3]:
from rice_Ml.preprocessing import undersample_majority

FEATURE_COLS = [
    "rms", "zcr", "centroid", "bandwidth", "rolloff",
    *[f"mfcc_{i}" for i in range(1, 14)],
]
FEATURE_NAMES = [
    "RMS", "ZCR", "Centroid", "Bandwidth", "Rolloff",
    *[f"MFCC {i}" for i in range(1, 14)],
]
N_TOP_NOTES = 12

print("Downloading full GuitarSet dataset (360 tracks, ~855 MB, cached after first run) …")
df = load_guitarset(subset=False)
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

# Identify top-12 most frequent non-silence notes
top_notes = (
    df[df["midi_label"] != 0]["midi_label"]
    .value_counts()
    .head(N_TOP_NOTES)
    .index.tolist()
)
print(f"Top-{N_TOP_NOTES} MIDI notes: {sorted(top_notes)}")

# Keep silence + top-12 notes only
keep_labels = [0] + top_notes
df_filt = df[df["midi_label"].isin(keep_labels)].copy()
print(f"After filtering: {len(df_filt):,} rows ({len(df_filt)/len(df):.1%} of dataset)")

# Remap to contiguous integers: silence=0, notes sorted by frequency → 1..12
note_to_class = {note: i + 1 for i, note in enumerate(top_notes)}
note_to_class[0] = 0  # silence stays 0
df_filt["class_label"] = df_filt["midi_label"].map(note_to_class)

# --- Balance: cap silence at 2× the most frequent voiced class ---
df_balanced = undersample_majority(
    df_filt, label_col="class_label", majority_label=0,
    cap_multiplier=2.0, random_state=SEED,
)
print(f"\nClass balancing:")
print(f"  Before: {len(df_filt):,} rows, silence = {(df_filt['class_label']==0).mean():.1%}")
print(f"  After:  {len(df_balanced):,} rows, silence = {(df_balanced['class_label']==0).mean():.1%}")

X = df_balanced[FEATURE_COLS].to_numpy(dtype=float)
y = df_balanced["class_label"].to_numpy(dtype=int)
N_CLASSES = len(np.unique(y))

print(f"\nX shape : {X.shape}")
print(f"Classes : {N_CLASSES}  (0=silence, 1–{N_CLASSES-1}=MIDI notes)")
print(f"\nClass distribution (after balancing):")
for cls in range(N_CLASSES):
    n = (y == cls).sum()
    label = "silence" if cls == 0 else f"MIDI {top_notes[cls-1]}"
    print(f"  Class {cls:>2} ({label:<10}): {n:>8,}  ({n/len(y):.1%})")

ImportError: cannot import name 'undersample_majority' from 'rice_Ml.preprocessing' (/usr/local/lib/python3.12/dist-packages/rice_Ml/preprocessing/__init__.py)

## 5. Exploratory Data Analysis

In [ ]:
# --- 5.1 Class distribution (after balancing) ---
fig, ax = plt.subplots(figsize=(12, 4))
class_labels = ["Silence"] + [f"MIDI {top_notes[i]}" for i in range(N_TOP_NOTES)]
counts = [(y == c).sum() for c in range(N_CLASSES)]
colors = ["#4C72B0"] + ["#DD8452"] * N_TOP_NOTES
ax.bar(class_labels, counts, color=colors, alpha=0.85)
ax.set_title("Class Distribution (after balancing — silence capped at 2× largest note class)", fontweight="bold")
ax.set_ylabel("Frame count")
ax.set_xticklabels(class_labels, rotation=45, ha="right", fontsize=9)
for i, (patch, count) in enumerate(zip(ax.patches, counts)):
    ax.text(patch.get_x() + patch.get_width()/2, count + len(y)*0.003,
            f"{count/len(y):.1%}", ha="center", fontsize=7)
plt.tight_layout()
plt.show()

**Interpretation:** After balancing, silence is capped at ~22% of frames rather than 82%. Each of the 12 note classes is now comparable in size to the silence class, which means gradient updates are distributed more evenly across all classes during training. Without this step, a model predicting "silence" for every input would achieve 82% accuracy while learning nothing about individual notes.

In [ ]:
# --- 5.2 Mean feature values per class (silence vs. voiced aggregate) ---
silence_means = X[y == 0].mean(axis=0)
voiced_means  = X[y != 0].mean(axis=0)

x_pos = np.arange(len(FEATURE_NAMES))
fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(x_pos - 0.2, silence_means, width=0.4, label="Silence", color="#4C72B0", alpha=0.85)
ax.bar(x_pos + 0.2, voiced_means,  width=0.4, label="Any note", color="#DD8452", alpha=0.85)
ax.set_xticks(x_pos)
ax.set_xticklabels(FEATURE_NAMES, rotation=45, ha="right", fontsize=9)
ax.set_title("Feature Means: Silence vs. Any Voiced Frame", fontweight="bold")
ax.set_ylabel("Mean value")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- 5.3 MFCC heatmap: mean MFCC per class ---
mfcc_cols = [f"mfcc_{i}" for i in range(1, 14)]
mfcc_idx  = [FEATURE_COLS.index(c) for c in mfcc_cols]
mfcc_means = np.array([X[y == c][:, mfcc_idx].mean(axis=0) for c in range(N_CLASSES)])

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(mfcc_means.T, aspect="auto", cmap="RdBu_r")
ax.set_yticks(range(13))
ax.set_yticklabels([f"MFCC {i}" for i in range(1, 14)], fontsize=8)
ax.set_xticks(range(N_CLASSES))
ax.set_xticklabels(class_labels, rotation=45, ha="right", fontsize=8)
ax.set_title("Mean MFCC Value per Class", fontweight="bold")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

**Interpretation:** The MFCC heatmap shows that:
- Silence (class 0) has a distinctly different low-frequency energy profile (MFCC 1–3) from all note classes.
- Higher MIDI notes (right side) tend to have more energy in higher MFCCs (finer spectral detail) compared to lower notes.
- Several adjacent note classes look visually similar — the MLP must learn to exploit small differences across all 13 MFCCs simultaneously, which is why a nonlinear multi-layer model is needed.

## 6. Preprocessing

Three steps:
1. **Stratified 80/20 train/test split** — preserves class proportions in both splits.
2. **Standardise** — zero mean, unit variance per feature, fit on training data only.
3. **One-hot encode labels** — the MLP outputs a score per class; MSE loss requires targets in $\{0,1\}^K$.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit(X_train).transform(X_train)
X_test_sc  = scaler.transform(X_test)

def to_onehot(labels, n_classes):
    oh = np.zeros((len(labels), n_classes))
    oh[np.arange(len(labels)), labels] = 1.0
    return oh

y_train_oh = to_onehot(y_train, N_CLASSES)
y_test_oh  = to_onehot(y_test,  N_CLASSES)

print(f"Train : {X_train_sc.shape[0]:,} samples")
print(f"Test  : {X_test_sc.shape[0]:,} samples")
print(f"y_train_oh shape: {y_train_oh.shape}")

## 7. K-Fold Cross-Validation

5-fold CV on a **stratified 100 K-frame subsample** of the balanced dataset assesses model stability without training 5 full-scale models. The subsample is drawn proportionally from each class so each fold mirrors the balanced class distribution.

In [ ]:
# Stratified subsample: 100 K frames, proportional per class
SUBSAMPLE = 100_000
sub_idx = []
for cls in range(N_CLASSES):
    cls_idx = np.where(y == cls)[0]
    n_take = max(1, int(SUBSAMPLE * len(cls_idx) / len(y)))
    sub_idx.extend(rng.choice(cls_idx, min(n_take, len(cls_idx)), replace=False))
sub_idx = np.array(sub_idx)
rng.shuffle(sub_idx)

X_sub, y_sub = X[sub_idx], y[sub_idx]
sc_sub = StandardScaler()
X_sub_sc = sc_sub.fit(X_sub).transform(X_sub)
y_sub_oh  = to_onehot(y_sub, N_CLASSES)

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
cv_acc, cv_f1 = [], []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_sub_sc)):
    p = MLP(
        hidden_layers=[128, 64],
        activation=ReLU(),
        output_activation=ReLU(),
        loss=MeanSquaredError(),
        optimizer=SGD(learning_rate=0.01),
        n_epochs=10,
        batch_size=512,
        random_state=SEED,
    )
    p.fit(X_sub_sc[tr_idx], y_sub_oh[tr_idx])
    y_hat = np.argmax(p.predict(X_sub_sc[val_idx]), axis=1)
    cv_acc.append(accuracy(y_sub[val_idx], y_hat))
    cv_f1.append(f1_score(y_sub[val_idx], y_hat))
    print(f"  Fold {fold+1}: acc={cv_acc[-1]:.4f}  f1={cv_f1[-1]:.4f}")

mean_acc = sum(cv_acc) / len(cv_acc)
std_acc  = (sum((v - mean_acc)**2 for v in cv_acc) / len(cv_acc))**0.5
mean_f1  = sum(cv_f1)  / len(cv_f1)
std_f1   = (sum((v - mean_f1)**2  for v in cv_f1)  / len(cv_f1))**0.5
print(f"\nCV Accuracy : {mean_acc:.4f} ± {std_acc:.4f}")
print(f"CV F1       : {mean_f1:.4f} ± {std_f1:.4f}")

**Interpretation:** Low standard deviation across folds means the MLP learns a stable boundary regardless of which frames are held out. High variance would indicate the 100 K subsample is too small or the model is overfitting to specific frame sequences.

## 8. Train on Full Dataset

In [ ]:
model = MLP(
    hidden_layers=[128, 64],
    activation=ReLU(),
    output_activation=ReLU(),
    loss=MeanSquaredError(),
    optimizer=SGD(learning_rate=0.01),
    n_epochs=30,
    batch_size=512,
    random_state=SEED,
)

print(f"Training on {X_train_sc.shape[0]:,} samples, {model.n_epochs} epochs, batch_size={model.batch_size} …")
model.fit(X_train_sc, y_train_oh)
print("Training complete.")

In [ ]:
# --- Loss curve ---
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(model.loss_history_) + 1), model.loss_history_,
        linewidth=2, color="#4C72B0")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("MLP Training Loss Curve", fontweight="bold")
ax.set_yscale("log")
plt.tight_layout()
plt.show()
print(f"Initial loss: {model.loss_history_[0]:.4f}  →  Final loss: {model.loss_history_[-1]:.4f}")

**Interpretation:** A monotonically decreasing loss confirms that backpropagation is working correctly and the model is learning. Log scale reveals whether improvement stalls early (plateau) or continues steadily throughout training. If the final loss is still falling, more epochs would help.

## 9. Evaluate

In [ ]:
y_raw  = model.predict(X_test_sc)          # shape (n, 13)
y_pred = np.argmax(y_raw, axis=1)          # class labels 0–12

acc = accuracy(y_test, y_pred)
f1  = f1_score(y_test, y_pred)

print(f"Test Accuracy : {acc:.4f}")
print(f"Test F1       : {f1:.4f}")
print()

# Per-class breakdown
print(f"{'Class':<18} {'Support':>9} {'Precision':>10} {'Recall':>8} {'F1':>8}")
print("-" * 58)
for cls in range(N_CLASSES):
    label = "Silence" if cls == 0 else f"MIDI {top_notes[cls-1]}"
    mask_true = (y_test == cls)
    mask_pred = (y_pred == cls)
    if mask_true.sum() == 0:
        continue
    p = precision(y_test == cls, y_pred == cls) if mask_pred.sum() > 0 else 0.0
    r = recall(y_test == cls, y_pred == cls)
    f = f1_score(y_test == cls, y_pred == cls) if (p + r) > 0 else 0.0
    print(f"  {label:<16} {mask_true.sum():>9,} {p:>10.4f} {r:>8.4f} {f:>8.4f}")

In [ ]:
# --- Confusion matrix (normalised by true class) ---
cm = np.zeros((N_CLASSES, N_CLASSES), dtype=int)
for t, p in zip(y_test, y_pred):
    cm[t, p] += 1

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(N_CLASSES))
ax.set_yticks(range(N_CLASSES))
ax.set_xticklabels(class_labels, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(class_labels, fontsize=8)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix (row-normalised)", fontweight="bold")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

**Interpretation:** The diagonal shows per-class recall. Off-diagonal entries show which classes the model confuses. We expect:
- Silence (class 0) to have high recall — it's the majority class and energetically distinct.
- Adjacent MIDI notes to be most confused with each other — they share harmonic content and differ only in fine spectral detail.

## 9.1 Comparison to Perceptron: Voiced/Silent Detection

To directly compare with the Perceptron notebook, we collapse the MLP's 13-class predictions into binary voiced/silent — any predicted non-zero class counts as "voiced". This shows whether the richer model also improves on the task the Perceptron was designed for.

In [ ]:
y_test_binary = (y_test != 0).astype(int)
y_pred_binary = (y_pred != 0).astype(int)

bin_acc  = accuracy(y_test_binary, y_pred_binary)
bin_prec = precision(y_test_binary, y_pred_binary)
bin_rec  = recall(y_test_binary, y_pred_binary)
bin_f1   = f1_score(y_test_binary, y_pred_binary)

print("Derived binary voiced/silent performance (MLP, full dataset):")
print(f"{'Metric':<12} {'MLP (13-class)':>16} {'Perceptron baseline':>22}")
print("-" * 52)
perc_scores = {"Accuracy": 0.9622, "Precision": 0.8799, "Recall": 0.7146, "F1": 0.7887}
for name, val in [("Accuracy", bin_acc), ("Precision", bin_prec), ("Recall", bin_rec), ("F1", bin_f1)]:
    print(f"{name:<12} {val:>16.4f} {perc_scores[name]:>22.4f}")

**Interpretation:** The MLP was trained on 13-class MIDI classification, not directly on the voiced/silent binary task. Collapsing its predictions to binary is a sanity check, not a fair comparison — the Perceptron's single binary boundary is purpose-built for this task. The MLP's advantage is that it classifies *which* note is playing, not just *whether* a note is playing.

## 10. Sklearn Comparison

In [ ]:
sk_model = SklearnMLP(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="sgd",
    learning_rate_init=0.01,
    max_iter=30,
    batch_size=512,
    random_state=SEED,
)
print("Fitting sklearn MLPClassifier …")
sk_model.fit(X_train_sc, y_train)
y_pred_sk = sk_model.predict(X_test_sc)

sk_acc = accuracy(y_test, y_pred_sk)
sk_f1  = f1_score(y_test, y_pred_sk)

print(f"\n{'Metric':<12} {'From Scratch':>14} {'Sklearn':>10}")
print("-" * 38)
print(f"{'Accuracy':<12} {acc:>14.4f} {sk_acc:>10.4f}")
print(f"{'F1':<12} {f1:>14.4f} {sk_f1:>10.4f}")

**Interpretation:** Sklearn's MLP uses Adam + cross-entropy by default which is more numerically stable than SGD + MSE, so it typically outperforms our implementation. The comparison validates that our model is in the right ballpark — large discrepancies would indicate a bug in our backpropagation or weight updates.

## 11. Architecture Exploration

We compare three hidden layer configurations on the 100 K subsample to understand the bias-variance tradeoff. Deeper/wider networks have more capacity but risk overfitting.

In [ ]:
configs = {
    "[64]":         [64],
    "[128, 64]":    [128, 64],
    "[256, 128, 64]": [256, 128, 64],
}

# Use a fixed 80/20 split of the subsample for speed
X_atr, X_aval, y_atr, y_aval = train_test_split(
    X_sub_sc, y_sub, test_size=0.2, random_state=SEED, stratify=y_sub
)
y_atr_oh = to_onehot(y_atr, N_CLASSES)

fig, ax = plt.subplots(figsize=(9, 5))
results = {}
for name, layers in configs.items():
    m = MLP(
        hidden_layers=layers,
        activation=ReLU(),
        output_activation=ReLU(),
        loss=MeanSquaredError(),
        optimizer=SGD(learning_rate=0.01),
        n_epochs=20,
        batch_size=512,
        random_state=SEED,
    )
    m.fit(X_atr, y_atr_oh)
    val_pred = np.argmax(m.predict(X_aval), axis=1)
    val_acc  = accuracy(y_aval, val_pred)
    val_f1   = f1_score(y_aval, val_pred)
    results[name] = {"acc": val_acc, "f1": val_f1}
    ax.plot(range(1, len(m.loss_history_) + 1), m.loss_history_, label=name, linewidth=2)
    print(f"{name:<20}  val_acc={val_acc:.4f}  val_f1={val_f1:.4f}")

ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss (log)")
ax.set_yscale("log")
ax.set_title("Loss Curves by Architecture (100 K subsample)", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

**Interpretation:** Deeper/wider networks reach lower training loss but may not always give the best validation F1. If the largest architecture gets the lowest loss but similar F1 to the medium one, it is beginning to overfit — memorising the training frames rather than generalising. The medium `[128, 64]` configuration is a good default for this feature space and dataset size.

## 12. Summary

| | |
|---|---|
| **Algorithm** | Multi-Layer Perceptron (backprop + mini-batch SGD) |
| **Task** | 13-class MIDI note classification (silence + top-12 notes) |
| **Features** | 18 audio features per 46 ms frame (RMS, ZCR, spectral, MFCCs) |
| **Training data** | All 360 GuitarSet tracks, all 6 strings — silence undersampled to 2× largest note class |
| **Architecture** | [128, 64] hidden units, ReLU activations, MSE loss |
| **Hyperparameters** | η = 0.01, batch = 512, epochs = 30 |

**Takeaways:**

1. **Class imbalance is the primary challenge.** 82% silence in raw data causes a naive model to predict silence for everything. Undersampling silence to 2× the largest note class gives all classes meaningful gradient signal without discarding voiced frames.
2. The MLP learns non-linear boundaries the Perceptron cannot — the decreasing loss curve confirms it is exploiting structure in the data that a single hyperplane misses.
3. Adjacent MIDI notes share most of their harmonic content and are the hardest to separate. MFCC 2–13 carry most of the discriminative information between notes.
4. Sklearn's Adam + cross-entropy outperforms our SGD + MSE — implementing cross-entropy loss with softmax output is the natural next step for `rice_Ml`.